# ALMA Data Reduction Tutorial: CO(12-11) in W2246-0526 (Band 6)

In this tutorial you will:
- install and run CASA6 directly in Colab (no local install needed)
- image a real ALMA visibility dataset — a continuum map, then a spectral cube
- make a continuum map, a spectral-line profile, moment-0/-8 maps (Python) and a
  moment-1 map (CASA)
- (Part 2) compare line fluxes/ratios across several lines observed toward the same target

This follows a 30-minute intro talk on interferometry and line/continuum imaging —
beginning PhD students / postdocs, first time touching ALMA data.

**Target:** W2246-0526 (WISE J224607.57-052635.0), a hyperluminous obscured QSO at **z = 4.601**.
**Data:** a single ALMA Band 6 spectral window (native ~9.3 km/s channels, time-binned to 60s,
already calibrated) covering the **CO(12-11)** transition (rest frequency 1381.995 GHz).

**Structure:**
1. Setup — install modular CASA6 and fetch the demo data
2. Part 1 — image *one* visibility dataset: continuum, then a spectral cube; make a
   continuum map, a spectral profile, moment-0/-8 (Python) and moment-1 (CASA)
2. Part 2 — read in cubes for the other FIR/CO lines covered by this target and compare
   line fluxes/ratios

This mirrors the standard modular-CASA6-in-Colab pattern from
[casangi/examples/community/casa6_demo.ipynb](https://colab.research.google.com/github/casangi/examples/blob/master/community/casa6_demo.ipynb).


## 0. Setup

Install the modular CASA6 packages (no full CASA install needed) and `gdown` to fetch data from Google Drive.

In [ ]:
%%capture
!pip install casatools==6.7.5.18 casatasks==6.7.5.18 casadata gdown


CASA6 needs its measures/reference-frame tables (Earth orientation parameters, etc.)
before `casatools`/`casatasks` can be imported. By default this is fetched at runtime
from ASTRON/NRAO by `casaconfig` — but that network fetch is unreliable from Colab (it
can time out partway, surfacing as `ImportError: measures data is not available`).
Sidestep it with `casadata`, a pip package that bundles the same tables directly (no
runtime download needed), pointing `measurespath` straight at it and turning off the
auto-update checks that would otherwise still try (and fail) to reach ASTRON/NRAO --
the same pattern CASA site installations use for a shared, pre-populated data dir:

In [ ]:
import os
import casadata

os.makedirs(os.path.expanduser("~/.casa"), exist_ok=True)
with open(os.path.expanduser("~/.casa/config.py"), "w") as f:
    f.write(f'measurespath = "{casadata.datapath}"\n')
    f.write("measures_auto_update = False\n")
    f.write("data_auto_update = False\n")

print("CASA data configured from casadata (no network fetch needed):", casadata.datapath)


Download the demo visibility data. Replace `DRIVE_FOLDER_URL` below with the shared Drive folder
link if it changes; `gdown.download_folder` pulls everything in that folder (the zipped MS).

In [ ]:
import gdown, glob, os, zipfile

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1BKiGlo9c2LY6AA5GPqFOMqArPFdSt1Tt?usp=sharing"

os.makedirs("data", exist_ok=True)
gdown.download_folder(url=DRIVE_FOLDER_URL, output="data", quiet=False, use_cookies=False)

zips = glob.glob("data/**/*.zip", recursive=True)
print("found zip(s):", zips)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall("data")

ms_dirs = glob.glob("data/**/*.ms", recursive=True)
vis = ms_dirs[0]
print("using MS:", vis)


## Part 1 — imaging a single visibility dataset

### 1.1 Orientation: what's in this Measurement Set?

In [ ]:
from casatasks import listobs

listobs(vis=vis, listfile="listobs.txt", overwrite=True)
!cat listobs.txt


A few things to note in the listing above:
- **Fields**: the sky positions observed (should be just our one target here).
- **Spectral Windows (spw)**: one spw, 240 channels, ~7.8 kHz (~9.3 km/s) wide each —
  this has already been split down from the full Band 6 setup to just the window
  covering our line of interest.
- **Scans**: separate observing blocks; `tclean`/`mstransform` can select scans, spws,
  fields, etc. independently.

The line targeted here is **CO(12-11)**, rest frequency 1381.995 GHz. At the systemic
redshift z = 4.601 this should appear at an observed frequency of
1381.995 / (1 + 4.601) = **246.75 GHz** — inside this spw's coverage. Keep that number in
mind for the spectrum in step 1.4.

### 1.2 `tclean`: continuum image

A quick "dirty" (`niter=0`) multi-frequency-synthesis image — averages the whole spw into one continuum map.

In [ ]:
from casatasks import tclean, exportfits

os.system("rm -rf cont.*")
tclean(vis=vis, imagename="cont", specmode="mfs",
       imsize=512, cell="0.05arcsec", weighting="natural",
       niter=0, pblimit=0.0)
exportfits("cont.image", "cont.fits", overwrite=True)


### 1.3 `tclean`: spectral cube

Same field, but `specmode='cube'` keeps every channel separate instead of averaging
them.

**On `restfreq`:** we pass the *observed-frame* predicted line center (246.75 GHz),
not the literature rest-frame CO(12-11) frequency (1381.995 GHz). `restfreq` is what
CASA treats as "zero velocity" for the image's velocity axis — feeding it the
unredshifted rest frequency would make the velocity axis measure our line's huge
cosmological recession velocity (hundreds of thousands of km/s) instead of the small,
physically meaningful local kinematics (rotation, outflows, ~100s of km/s) we actually
want out of the moment-1 map in step 1.7.

In [ ]:
os.system("rm -rf cube.*")
tclean(vis=vis, imagename="cube", specmode="cube",
       imsize=512, cell="0.05arcsec", weighting="natural",
       niter=0, pblimit=0.0,
       restfreq="246.75GHz", outframe="LSRK")
exportfits("cube.image", "cube.fits", overwrite=True)


### 1.4 Plot the continuum map

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

cont_hdu = fits.open("cont.fits")[0]
cont_map = np.squeeze(cont_hdu.data)

plt.figure(figsize=(6, 5))
im = plt.imshow(cont_map * 1e3, origin="lower", cmap="inferno")
plt.colorbar(im, label="mJy/beam")
plt.title("W2246-0526 continuum, Band 6 (~247 GHz)")
plt.xlabel("pixel"); plt.ylabel("pixel")
plt.show()


### 1.5 Plot the spectral line profile

Spectrum at the map center (the source is at the phase center in this dataset).

In [ ]:
cube_hdul = fits.open("cube.fits")
cube = np.squeeze(cube_hdul[0].data)          # (nchan, ny, nx)
header = cube_hdul[0].header

nchan, ny, nx = cube.shape
crval3, cdelt3, crpix3 = header["CRVAL3"], header["CDELT3"], header["CRPIX3"]
freq_hz = crval3 + (np.arange(nchan) - (crpix3 - 1)) * cdelt3

cy, cx = ny // 2, nx // 2
spectrum_mjy = cube[:, cy, cx] * 1e3

plt.figure(figsize=(10, 4))
plt.step(freq_hz / 1e9, spectrum_mjy, where="mid", color="steelblue")
plt.axhline(0, color="gray", lw=0.6, ls="--")
plt.axvline(246.75, color="crimson", lw=0.8, ls=":",
            label="CO(12-11) @ z=4.601 (predicted)")
plt.xlabel("Frequency [GHz]"); plt.ylabel("Flux density [mJy/beam]")
plt.title("Spectrum at map center")
plt.legend()
plt.show()


### 1.6 Moment-0 and moment-8 maps (pure Python)

- **Moment-8** = peak signal-to-noise across the spectrum at each pixel — the standard
  quick "where's the line" detection map. Each channel is normalized by its own noise
  first, so a broad-but-faint line isn't drowned out by one noisy narrow channel.
- **Moment-0** = integrated line flux, summed over the channels that contain the line
  (identified from the spectrum above — adjust `line_lo`/`line_hi` if your channel
  range differs).

> **Fitting gotcha worth knowing:** if you later fit a Gaussian to a spectrum like this
> one, the *initial guess* for the line width matters a lot. A narrow guess (e.g. a
> couple of channels) can pull a `LevMarLSQFitter` fit onto a single noisy channel
> instead of the real, often much broader, line — you'd see a suspiciously narrow
> fitted FWHM (barely more than one channel) as the symptom. A guess closer to a
> realistic line width (e.g. ~300 km/s) plus a fit window wide enough to see the whole
> profile avoids this. This is a real failure mode we hit building this tutorial's own
> automated detection pipeline.

In [ ]:
# Line channel range read off the spectrum plot above. This spw's channels run from
# 248.0017 GHz (ch 0) downward in ~7.8125 kHz (~9.3 km/s) steps, so the predicted
# CO(12-11) center (246.75 GHz) falls around ch ~160; the window below gives it
# +/-400ish km/s of margin on each side -- adjust after inspecting your own plot.
line_lo, line_hi = 110, 200   # <-- adjust after inspecting your own spectrum plot

stds = np.nanstd(cube, axis=(1, 2))                  # per-channel RMS across the map
rms = stds.copy()
rms[(rms == 0) | np.isnan(rms)] = np.nan
snr_cube = cube / rms[:, np.newaxis, np.newaxis]
moment8 = np.nanmax(snr_cube, axis=0)                # peak-SNR map

chan_width_hz = abs(cdelt3)
restfreq_hz = 246.75e9   # observed-frame value used as tclean's restfreq in 1.3, not
                         # the literature rest-frame 1381.995 GHz -- see note there.
dv_kms = chan_width_hz / restfreq_hz * 2.99792458e5
moment0 = np.nansum(cube[line_lo:line_hi], axis=0) * dv_kms * 1e3   # mJy/beam * km/s

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
im0 = axes[0].imshow(moment0, origin="lower", cmap="inferno")
axes[0].set_title("Moment-0 (integrated flux)")
plt.colorbar(im0, ax=axes[0], label="mJy/beam km/s", shrink=0.85)

im1 = axes[1].imshow(moment8, origin="lower", cmap="inferno")
axes[1].set_title("Moment-8 (peak SNR)")
plt.colorbar(im1, ax=axes[1], label="SNR", shrink=0.85)
plt.tight_layout()
plt.show()


### 1.7 Moment-1 map (the CASA way)

Moment-1 (intensity-weighted velocity) needs a flux threshold/mask to avoid being
dominated by noise outside the line — CASA's `immoments` handles that natively, which
is why we reach for it here instead of a five-line numpy version.

`immoments` reads its velocity axis straight from `cube.image`'s header, i.e. from the
`restfreq` we set in 1.3 — that's why the map below should show a small range of
offsets (~ +/- 100s of km/s), not the line's raw cosmological recession velocity.

In [ ]:
from casatasks import immoments

os.system("rm -rf cube.mom1*")
immoments(imagename="cube.image", moments=[1],
          chans=f"{line_lo}~{line_hi - 1}",
          includepix=[3 * float(np.nanmedian(stds)), 1e9],   # crude 3-sigma-ish mask
          outfile="cube.mom1")
exportfits("cube.mom1", "cube.mom1.fits", overwrite=True)

mom1 = np.squeeze(fits.open("cube.mom1.fits")[0].data)
plt.figure(figsize=(6, 5))
im = plt.imshow(mom1, origin="lower", cmap="RdBu_r")
plt.colorbar(im, label="km/s")
plt.title("Moment-1 (velocity field)")
plt.show()


## Part 2 — comparing multiple lines for the same target

W2246-0526 has been observed across many ALMA bands, covering [OI]63um, [OIII]88um,
[NII]122um, [OI]145um, [CII]158um, [NII]205um, [CI]370um and [CI]609um (plus CO(12-11)
from Part 1). This section reads in pre-made cubes for those lines (as they become
available) and compares integrated fluxes.

We already have a second one: a Band 10 cube with a bright line around 846 GHz.
At z = 4.601, [OI]63um (rest 4744.777 GHz) predicts an observed frequency of
4744.777 / (1 + 4.601) = **847.1 GHz** — matching the ~846 GHz line almost exactly, so
that's our second line. Its FITS cube (`WISE2246-0526_12m_838GHz_tbin30s_cube.cube.fits`)
is in the same shared Drive folder as the Band 6 data, so it downloaded already back in
Setup — no extra fetch needed.

> **Status:** the remaining bands' cubes are still being produced by the main pipeline
> — add them to `LINE_CUBES` below as they're exported and uploaded to the same Drive
> folder.

In [ ]:
oi63_matches = glob.glob("data/**/WISE2246-0526_12m_838GHz_tbin30s_cube.cube.fits", recursive=True)
print("found:", oi63_matches)
oi63_path = oi63_matches[0]

LINE_CUBES = {
    "CO(12-11)": "cube.fits",     # from Part 1, already in hand
    "[OI]63um":  oi63_path,       # Band 10, downloaded in Setup
    # "[CII]158um":  "data/.../W2246-0526_CII158um_cube.fits",
    # "[NII]205um":  "data/.../W2246-0526_NII205um_cube.fits",
    # "[OI]145um":   "data/.../W2246-0526_OI145um_cube.fits",
    # ... add the rest once available
}

REST_GHZ = {
    "CO(12-11)": 1381.995,
    "[OI]63um": 4744.777490, "[OI]145um": 2060.069000,
    "[NII]122um": 2459.380000, "[NII]205um": 1461.133800,
    "[OIII]88um": 3393.006240, "[CII]158um": 1900.536900,
    "[CI]370um": 809.341970, "[CI]609um": 492.160651,
}
Z_SYSTEMIC = 4.601


In [ ]:
def integrated_flux_mjy_kms(fits_path, rest_ghz, z=Z_SYSTEMIC, pos=None, half_width_chan=20):
    """Sum flux around the map center (or `pos=(y,x)`) over +/- half_width_chan
    channels centered on the peak-SNR channel -- a simplified version of Part 1's
    moment-0, applied consistently to every line's cube.

    Uses the *observed-frame* frequency (rest_ghz / (1+z)) for the Hz->km/s
    conversion, matching how these cubes' own restfreq was set during imaging
    (see the note in step 1.3) -- using the literal rest-frame value here would
    under-report the flux by a factor of ~(1+z).
    """
    hdul = fits.open(fits_path)
    c = np.squeeze(hdul[0].data)
    hdr = hdul[0].header
    nchan = c.shape[0]
    cdelt3 = hdr["CDELT3"]

    stds = np.nanstd(c, axis=(1, 2))
    rms = stds.copy(); rms[(rms == 0) | np.isnan(rms)] = np.nan

    y, x = pos if pos else (c.shape[1] // 2, c.shape[2] // 2)
    spec = c[:, y, x] / rms
    peak_ch = int(np.nanargmax(spec))
    lo, hi = max(0, peak_ch - half_width_chan), min(nchan, peak_ch + half_width_chan)

    obs_ghz = rest_ghz / (1 + z)
    dv_kms = abs(cdelt3) / (obs_ghz * 1e9) * 2.99792458e5
    return float(np.nansum(c[lo:hi, y, x]) * dv_kms * 1e3)   # mJy/beam * km/s


fluxes = {name: integrated_flux_mjy_kms(path, REST_GHZ[name])
          for name, path in LINE_CUBES.items()}
print(fluxes)


In [ ]:
names = list(fluxes.keys())
vals = [fluxes[n] for n in names]
ref = names[0]   # normalize ratios to the first available line -- CO(12-11) here

plt.figure(figsize=(7, 4))
plt.bar(names, [v / fluxes[ref] for v in vals], color="steelblue")
plt.ylabel(f"Line flux / {ref}")
plt.title(f"Line ratios (z = {Z_SYSTEMIC})")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
